# Phase 15 — MLflow Tracking & Experiment Management
**ChurnOps MLOps Pipeline**

---

### Objectives
1. **Centralized Experiment Tracking**: Connect to the local SQLite-backed MLflow tracking server (`sqlite:///mlruns/mlflow.db`).
2. **Audit All Model Iterations**: Query and compare all 10 logged runs across:
   - Baseline models (Phase 9)
   - Tuned regularized models (Phase 12)
   - Blended & Stacking ensembles (Phase 13)
   - Production Champion (Phase 14)
3. **Reproducibility Verification**: Inspect parameters, metrics, tags, and logged visual artifacts directly via the MLflow API.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import mlflow
import pandas as pd
from IPython.display import display, HTML

from src.utils.config import PARAMS

mlflow.set_tracking_uri(PARAMS["mlflow"]["tracking_uri"])
exp_name = PARAMS["mlflow"]["experiment_name"]
experiment = mlflow.get_experiment_by_name(exp_name)

print("MLflow Tracking URI:", mlflow.get_tracking_uri())
print(f"Experiment: '{exp_name}' (ID: {experiment.experiment_id})")


---
## 1. Querying Logged Runs from MLflow SQLite Backend
We execute `mlflow.search_runs()` to extract the complete experiment history.


In [ ]:
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Filter and format key columns for clean inspection
display_cols = [
    "tags.mlflow.runName",
    "tags.model_stage",
    "metrics.val_f1",
    "metrics.val_roc_auc",
    "metrics.generalization_gap",
    "metrics.test_f1_default",
    "metrics.test_roc_auc",
    "status",
]
available_cols = [c for c in display_cols if c in runs_df.columns]
summary_table = runs_df[available_cols].copy()
summary_table.rename(columns={
    "tags.mlflow.runName": "Run Name",
    "tags.model_stage": "Stage",
    "metrics.val_f1": "CV Val F1",
    "metrics.val_roc_auc": "CV ROC-AUC",
    "metrics.generalization_gap": "Gen Gap",
    "metrics.test_f1_default": "Test F1",
    "metrics.test_roc_auc": "Test ROC-AUC",
    "status": "Run Status"
}, inplace=True)

display(HTML(summary_table.to_html(index=False)))
print(f"Total Runs Retrieved from MLflow DB: {len(runs_df)}")


---
## 2. Production Champion Run Audit
Examining the production champion run tagged `is_champion=true`.


In [ ]:
champ_run = runs_df[runs_df["tags.is_champion"] == "true"].iloc[0]

print("==================================================")
print(f"Champion Run ID:   {champ_run['run_id']}")
print(f"Champion Run Name: {champ_run['tags.mlflow.runName']}")
print(f"Model Family:      {champ_run['tags.model_family']}")
print(f"Stage:             {champ_run['tags.model_stage']}")
print(f"Status:            {champ_run['tags.status']}")
print("Logged Holdout Test Metrics:")
print(f"  Test ROC-AUC:    {champ_run.get('metrics.test_roc_auc', 'N/A'):.4f}")
print(f"  Test F1:         {champ_run.get('metrics.test_f1_default', 'N/A'):.4f}")
print(f"  Test Recall:     {champ_run.get('metrics.test_recall_default', 'N/A'):.4f}")
print(f"  Test Precision:  {champ_run.get('metrics.test_precision_default', 'N/A'):.4f}")
print(f"  Test Accuracy:   {champ_run.get('metrics.test_accuracy_default', 'N/A'):.4f}")
print("==================================================")


---
## 3. How to Launch MLflow UI
To open the interactive web UI and explore runs, parameters, metrics plots, and artifacts visually:

```powershell
.venv\Scripts\mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db --port 5000
```
Then navigate to `http://localhost:5000` in your browser.
